# NFL opening-drive neural experiment

Run this notebook in Google Colab. To use a GPU, select **Runtime → Change runtime type → GPU** before running the cells. The code uses the runtime's installed PyTorch and checks whether CUDA is available.

The experiment uses run or pass labels from eligible scrimmage decisions. The GRU can use earlier observed decisions on the same drive. First eligible plays give a separate measure with no prior drive history. Results use a chronological test period and describe a historical evaluation.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repository = "https://github.com/Soham-Padalkar/nfl-playcall-intelligence.git"
checkout = Path("/content/nfl-playcall-intelligence")
if not checkout.is_dir():
    subprocess.run(["git", "clone", repository, str(checkout)], check=True)
os.chdir(checkout)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

In [ ]:
import torch
from playcall.data import DataStore

seasons = [2022, 2023, 2024, 2025, 2026]
cutoff = "2026-11-23"

print("PyTorch", torch.__version__)
print(torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU")
sources = DataStore().refresh(seasons, supplemental=False)
print({name: record["status"] for name, record in sources.items()})

In [ ]:
cmd = [
    sys.executable, "-m", "playcall.neural",
    "--seasons", *[str(season) for season in seasons],
    "--cutoff", cutoff, "--device", "auto",
    "--batch-size", "128", "--accumulation-steps", "2",
]
subprocess.run(cmd, check=True)

In [ ]:
import json
import pandas as pd

report_path = Path("models/neural-results.json")
report = json.loads(report_path.read_text(encoding="utf-8"))
display(pd.DataFrame(report["splits"]).T)
display(pd.DataFrame({
    name: {
        "validation log loss": metrics["validation_log_loss"],
        "test log loss": metrics["opening_drives"]["log_loss"],
        "test Brier score": metrics["opening_drives"]["brier"],
        "test accuracy": metrics["opening_drives"]["accuracy"],
        "test first eligible play log loss": metrics["first_eligible_play"]["log_loss"],
        "test drive pass share MAE": metrics["opening_drives"]["drive_pass_share_mae"],
    }
    for name, metrics in report["metrics"].items()
}).T)
print("Selected by validation:", report["selected_by_validation_log_loss"])
print("Report and checkpoints:", report_path.resolve(), Path("models/neural").resolve())

The report includes the data file hashes, device, train settings and sample counts. A new nflverse download can include source revisions. The Colab runtime deletes its local files when the session ends. Use the Colab Files panel to download the report and checkpoints if you need them later.